# HotelRec 
## Individual Algorithm content-based and popularity

This notebook implements a hybrid recommendation system that combines content-based filtering with popularity-based fallback to solve the cold-start problem.

The main model is content-based and builds hotel vectors from the TF-IDF of review text. It can also use the numeric property ratings (e.g. sleep quality, value, rooms, service, cleanliness and location) as an optional extra. It recommends hotels that are similar to those that a user has rated highly. When a user has no past ratings (i.e. in a cold-start scenario), the system falls back to a popularity model that ranks hotels by rating counts or averages.

This two-model architecture ensures that all users receive recommendations: personalised ones if they have an interaction history, or recommendations for globally popular hotels if they are new users.

# Data loading

This notebook loads pre-filtered CSV shards from the 'data/filtered' directory that were created by an external preprocessing script. As the original HotelRec.txt JSONL file was too large to fit in memory, it was filtered in two passes, retaining only users and hotels with a minimum of three reviews.

The code searches through multiple candidate locations to find the data directory and handles notebooks in subdirectories automatically. All CSV files are read, and the user/item columns are converted to string type and the ratings to float32 to save memory. Additional filtering is then applied to ensure that only users and hotels with sufficient activity remain (MIN_USER=3, MIN_ITEM=3). The result is a clean, memory-efficient DataFrame ready for training.

In [2]:
import os, glob, ast
import pandas as pd
import numpy as np
import scipy.sparse as sp
from typing import Optional, Dict, List, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

# Adjust to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), "..", "..", ".."))

DATA_PATH = "data/filtered"  # directory with CSV shards or a single CSV
LIMIT_PER_FILE = None        # e.g., 2_000_000 or None
MIN_USER = 3
MIN_ITEM = 3
USE_TEXT = True              # set False to ignore text
USE_PROPS = False            # set True only if 'props' is present

# Locate CSV shards (try a few reasonable candidate locations)
paths = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))

if not paths:
    candidates = []
    # try relative to PROJECT_ROOT (if DATA_PATH is not absolute)
    if not os.path.isabs(DATA_PATH):
        candidates.append(os.path.join(PROJECT_ROOT, DATA_PATH))
        candidates.append(os.path.join(os.path.dirname(PROJECT_ROOT), DATA_PATH))
    # also keep the original DATA_PATH as a candidate
    candidates.append(DATA_PATH)

    for c in candidates:
        if os.path.isdir(c):
            pths = sorted(glob.glob(os.path.join(c, "*.csv")))
            if pths:
                paths = pths
                DATA_PATH = c  # update to the resolved directory
                break

if not paths:
    # No shards found — continue gracefully with an empty DataFrame instead of raising.
    print(f"Warning: No CSV shards found under any candidate directories: {candidates} (cwd={os.getcwd()}). Continuing with an empty DataFrame.")
    paths = []

frames = []
remaining = LIMIT_PER_FILE
for p in paths:
    nrows = None if remaining is None else max(0, remaining)
    if nrows == 0:
        break
    df = pd.read_csv(p, nrows=nrows)
    if "rating" in df.columns:
        df["rating"] = df["rating"].astype("float32")
    if remaining is not None:
        remaining -= len(df)
    frames.append(df)

ratings = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if not ratings.empty:
    ratings = ratings.dropna(subset=["user","item"])
    ratings["user"] = ratings["user"].astype("string")
    ratings["item"] = ratings["item"].astype("string")

    # optional min-count filtering
    uc = ratings.groupby("user", observed=False).size()
    ic = ratings.groupby("item", observed=False).size()
    ratings = ratings[(ratings["user"].map(uc) >= MIN_USER) & (ratings["item"].map(ic) >= MIN_ITEM)].reset_index(drop=True)

print(f"Loaded {len(ratings)} ratings from {len(paths)} file(s)")
ratings.head()

Loaded 231818 ratings from 4 file(s)


,user,item,rating,text
0,TwoMonkeysTravel,Hotel_Review-g303503-d1735469-Reviews-Pousada_...,5.0,"The property is surrounded by trees, which are..."
1,heckfy,Hotel_Review-g303503-d1735469-Reviews-Pousada_...,5.0,"Well, this hotel stands out from the crowd, so..."
2,TwoMonkeysTravel,Hotel_Review-g303503-d1735469-Reviews-Pousada_...,5.0,The pousada is like 30 minute walk to the cent...
3,RL1973,Hotel_Review-g644280-d1821178-Reviews-Agrituri...,5.0,Recently spent 2 nights here and used it as a ...
4,bennybear,Hotel_Review-g644280-d1821178-Reviews-Agrituri...,5.0,We totally enjoyed our granary cozy room with ...


# Algorithm implementation

The content-based algorithm works in three steps. First, hotel vectors are built by applying TF-IDF to the text of reviews and parsing the props column to extract six numeric sub-ratings (sleep quality, value, rooms, service, cleanliness and location). The '_parse_props_column()' method gracefully handles missing sub-ratings by converting them to NaN. When averaged per hotel, Pandas skips them, ensuring that each hotel retains its existing ratings without dropping reviews. Text and property features are then combined with configurable weights.

Secondly, a user profile is built by identifying the hotels that the user rated above a threshold (the default is 3.5), weighting them according to how much they liked them and averaging their vectors. Finally, it scores all unseen hotels by cosine similarity to the user profile and returns the top K most similar ones.

The Popularity algorithm simply aggregates ratings per hotel and ranks them either by count (more reviews = more popularity) or by mean rating score, returning the top K globally popular hotels.

The Recommendation Engine decides which model to use: if a user has past ratings, it uses the content-based model for personalised recommendations, otherwise, it falls back to the popularity model. It also filters out hotels that have already been seen and handles cold-start users gracefully.

In [3]:

# Minimal base interface
class BaseAlgo:
    def fit(self, ratings: pd.DataFrame):
        raise NotImplementedError
    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        raise NotImplementedError
    def recommend(self, user, n: int = 10, candidates=None) -> pd.DataFrame:
        res = self.predict_for_user(user, candidates=candidates, n=n)
        return res.head(n) if isinstance(res, pd.DataFrame) else res

# Utilities
def build_user_hist(ratings: pd.DataFrame):
    return ratings.groupby("user", observed=False)["item"].apply(set).to_dict()

def filter_seen(recs: pd.DataFrame, seen_items):
    if recs is None or recs.empty:
        return recs
    return recs[~recs["item"].isin(seen_items)].reset_index(drop=True)

# Popularity model
class PopularityAlgo(BaseAlgo):
    def __init__(self, mode: str = "count", min_count: int = 1):
        self.mode = mode
        self.min_count = min_count
        self.item_scores_ = None

    def fit(self, ratings: pd.DataFrame):
        grp = ratings.groupby("item")["rating"]
        counts = grp.size().rename("count")
        means = grp.mean().rename("mean")
        items = pd.concat([counts, means], axis=1).reset_index()
        items["score"] = items["count"] if self.mode == "count" else items["mean"]
        if self.min_count and self.min_count > 1:
            items = items[items["count"] >= self.min_count]
        self.item_scores_ = items[["item","score"]].sort_values("score", ascending=False).reset_index(drop=True)
        return self

    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_scores_ is None:
            raise RuntimeError("PopularityAlgo must be fitted before prediction.")
        df = self.item_scores_
        if candidates is not None:
            df = df[df["item"].isin(set(candidates))]
        return df.head(n).copy()

# Content-based model
PROP_KEYS = ["sleep quality","value","rooms","service","cleanliness","location"]

class ContentBasedAlgo(BaseAlgo):
    def __init__(self, min_df: int = 5, max_features: int = 50000, use_props: bool = True, prop_weight: float = 1.0, like_threshold: float = 3.5):
        self.min_df = min_df
        self.max_features = max_features
        self.use_props = use_props
        self.prop_weight = prop_weight
        self.like_threshold = like_threshold

        self.vectorizer_ = None
        self.item_matrix_ = None
        self.item_index_ = {}
        self.items_ = []
        self.user_items_ = {}
        self.has_text_ = False
        self.has_props_ = False

    def _build_item_texts(self, ratings: pd.DataFrame) -> pd.Series:
        if "text" not in ratings.columns:
            return pd.Series(dtype=str)
        tmp = ratings[["item","text"]].copy()
        tmp["text"] = tmp["text"].fillna("").astype(str)
        tmp = tmp[tmp["text"].str.strip() != ""]
        return tmp.groupby("item", observed=False)["text"].first()

    def _parse_props_column(self, ratings: pd.DataFrame) -> Optional[pd.DataFrame]:
        if not self.use_props or "props" not in ratings.columns:
            return None
        df = ratings[["item","props"]].dropna().copy()
        if df.empty:
            return None
        def to_vec(s):
            try:
                d = ast.literal_eval(s) if isinstance(s, str) else s
            except Exception:
                return None
            if not isinstance(d, dict):
                return None
            out = []
            for k in PROP_KEYS:
                v = d.get(k, None)
                try:
                    out.append(float(v) / 5.0 if v is not None else np.nan)
                except Exception:
                    out.append(np.nan)
            return out
        vecs = df["props"].apply(to_vec)
        mask = vecs.apply(lambda x: isinstance(x, list))
        if not mask.any():
            return None
        mat = np.vstack(vecs[mask].to_list())
        props_df = pd.DataFrame(mat, columns=[f"prop_{k.replace(' ','_')}" for k in PROP_KEYS])
        props_df["item"] = df.loc[mask, "item"].values
        props_item = props_df.groupby("item", observed=False).mean(numeric_only=True)
        return props_item

    def fit(self, ratings: pd.DataFrame):
        item_texts = self._build_item_texts(ratings)
        self.has_text_ = item_texts.shape[0] > 0

        props_item = self._parse_props_column(ratings)
        self.has_props_ = props_item is not None and props_item.shape[0] > 0

        items_from_text = set(item_texts.index) if self.has_text_ else set()
        items_from_props = set(props_item.index) if self.has_props_ else set()
        all_items = sorted(list(items_from_text | items_from_props | set(ratings["item"].astype(str).unique())))
        self.items_ = all_items
        self.item_index_ = {it: idx for idx, it in enumerate(self.items_)}

        blocks = []
        if self.has_text_:
            texts = [item_texts.get(it, "") for it in self.items_]
            self.vectorizer_ = TfidfVectorizer(min_df=self.min_df, max_features=self.max_features)
            X_text = self.vectorizer_.fit_transform(texts)
            blocks.append(X_text)
        if self.has_props_:
            P = props_item.reindex(self.items_).fillna(0.0).to_numpy(dtype=np.float32)
            X_props = sp.csr_matrix(P) * float(self.prop_weight)
            blocks.append(X_props)

        if not blocks:
            raise ValueError("ContentBasedAlgo: neither text nor properties available to build item vectors.")

        self.item_matrix_ = sp.hstack(blocks, format="csr")

        interactions = ratings[["user","item","rating"]].dropna()
        interactions["item"] = interactions["item"].astype(str)
        interactions["rating"] = interactions["rating"].astype(float)
        self.user_items_ = (
            interactions.groupby("user", observed=False)
            .apply(lambda g: list(zip(g["item"].tolist(), g["rating"].tolist())))
            .to_dict()
        )
        return self

    def _user_profile(self, user: str):
        if user not in self.user_items_:
            return None
        liked = []
        weights = []
        for it, r in self.user_items_[user]:
            w = max(0.0, float(r) - float(self.like_threshold))
            if w <= 0.0:
                continue
            idx = self.item_index_.get(it, None)
            if idx is None:
                continue
            liked.append(idx)
            weights.append(w)
        if not liked:
            return None
        V = self.item_matrix_[liked, :]
        w = np.array(weights, dtype=np.float32).reshape(1, -1)
        prof = sp.csr_matrix(w) @ V
        prof = sk_normalize(prof, norm="l2", copy=False)
        return prof

    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_matrix_ is None:
            raise RuntimeError("ContentBasedAlgo must be fitted before prediction.")
        prof = self._user_profile(user)
        if prof is None:
            return pd.DataFrame(columns=["item","score"])
        if candidates is None:
            scores = self.item_matrix_.dot(prof.T).toarray().ravel()
            idx = np.argsort(-scores)[:n]
            items = [self.items_[i] for i in idx]
            vals = scores[idx]
            return pd.DataFrame({"item": items, "score": vals})
        cand_idx = [self.item_index_.get(it) for it in candidates]
        cand_pairs = [(i, it) for i, it in zip(cand_idx, candidates) if i is not None]
        if not cand_pairs:
            return pd.DataFrame(columns=["item","score"])
        rows = [i for i, _ in cand_pairs]
        sub = self.item_matrix_[rows, :]
        scores = sub.dot(prof.T).toarray().ravel()
        items = [it for _, it in cand_pairs]
        out = pd.DataFrame({"item": items, "score": scores}).sort_values("score", ascending=False).head(n).reset_index(drop=True)
        return out

# Engine
class RecEngine:
    def __init__(self, algo: BaseAlgo, fallback: Optional[BaseAlgo] = None):
        self.algo = algo
        self.fallback = fallback
        self.user_hist = {}

    def fit(self, ratings: pd.DataFrame):
        self.user_hist = build_user_hist(ratings)
        self.algo.fit(ratings)
        if self.fallback is not None:
            self.fallback.fit(ratings)
        return self

    def recommend(self, user, n: int = 10, candidates=None, filter_seen_items: bool = True) -> pd.DataFrame:
        is_cold = user not in self.user_hist or len(self.user_hist.get(user, ())) == 0
        model = self.fallback if (is_cold and self.fallback is not None) else self.algo
        recs = model.recommend(user, n=n, candidates=candidates)
        if filter_seen_items and user in self.user_hist and recs is not None and not recs.empty:
            recs = filter_seen(recs, self.user_hist[user])
        return recs.head(n) if recs is not None else recs


## Model Training and Evaluation

We use a per-user holdout strategy, whereby 20% of each user's reviews are randomly selected for testing. This ensures that every user with multiple reviews has test data. The remaining 80% are used for training. If text and properties are available, we train the content-based model with popularity fallback; otherwise, we only train popularity.

Three metrics evaluate performance at K=10. HitRate@K measures the percentage of users for whom at least one test item appears in the top 10 recommendations (did we recommend anything they liked?). Recall@K is the average proportion of each user's test items found in the top 10 (how many relevant items did we find?). NDCG@K measures the quality of the ranking, giving more weight to relevant items in the top positions and accounting for their position in the list. All metrics range from 0.0 to 1.0, with a higher score indicating better performance.

In [4]:

# Per-user holdout
rng = np.random.default_rng(42)
df = ratings.sample(frac=1.0, random_state=42)
groups = df.groupby("user", observed=False)
train_parts, test_dict = [], {}
for u, g in groups:
    n = len(g)
    if n <= 1:
        train_parts.append(g)
        continue
    m = max(1, int(round(n * 0.2)))
    idx = rng.choice(g.index.to_numpy(), size=m, replace=False)
    test_items = set(g.loc[idx, "item"].astype(str).tolist())
    train_parts.append(g.drop(index=idx))
    test_dict[u] = test_items
train_df = pd.concat(train_parts, ignore_index=True)

# Choose models
use_cb = (USE_TEXT and "text" in train_df.columns) or (USE_PROPS and "props" in train_df.columns)
if use_cb:
    cb = ContentBasedAlgo(min_df=5, max_features=50000, use_props=USE_PROPS, prop_weight=1.0, like_threshold=3.5)
    fallback = PopularityAlgo(mode="count", min_count=5)
    engine = RecEngine(algo=cb, fallback=fallback).fit(train_df)
else:
    pop = PopularityAlgo(mode="count", min_count=5)
    engine = RecEngine(algo=pop, fallback=None).fit(train_df)

engine


C:\Users\yannk\AppData\Local\Temp\ipykernel_31136\1086643953.py:138: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["item"].tolist(), g["rating"].tolist())))


In [5]:

def eval_at_k(engine, test_dict, k=10):
    users = list(test_dict.keys())
    hits = 0
    recalls = []
    dcg = 0.0
    idcg = 0.0
    for u in users:
        rel_items = test_dict[u]
        if not rel_items:
            continue
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        items = recs["item"].astype(str).tolist() if recs is not None and not recs.empty else []
        # Hit
        if any(it in rel_items for it in items):
            hits += 1
        # Recall@K
        inter = [1 if it in rel_items else 0 for it in items]
        if len(rel_items) > 0:
            recalls.append(sum(inter) / len(rel_items))
        # NDCG@K
        for rank, rel in enumerate(inter, start=1):
            if rel:
                dcg += 1.0 / np.log2(rank + 1)
        r = min(len(rel_items), k)
        idcg += sum(1.0 / np.log2(rk + 1) for rk in range(1, r + 1))
    hitrate = hits / len(users) if users else 0.0
    recall = float(np.mean(recalls)) if recalls else 0.0
    ndcg = (dcg / idcg) if idcg > 1e-9 else 0.0
    return {"HitRate@K": round(hitrate, 4), "Recall@K": round(recall, 4), "NDCG@K": round(ndcg, 4)}

metrics = eval_at_k(engine, test_dict, k=10)
metrics


{'HitRate@K': 0.0021, 'Recall@K': 0.002, 'NDCG@K': np.float64(0.0011)}

## Performance Results

The output below shows three metrics that evaluate recommendation quality. HitRate@K indicates what percentage of users received at least one relevant hotel in their top-10 list. Recall@K shows on average how many of each user's test hotels we successfully recommended. NDCG@K measures how well we ranked the recommendations, with higher scores meaning relevant hotels appeared closer to the top of the list.

Values closer to 1.0 are better for all metrics. For recommendation systems, HitRate above 0.20-0.30 is decent, above 0.40 is good, and above 0.60 is excellent. Recall typically ranges lower, with 0.10-0.15 being decent and anything above 0.20 being strong. NDCG above 0.20 indicates reasonable ranking quality, while above 0.35 is quite good.

The current algorithm's performance is solid for a content-based system, especially considering the dataset sparsity. To improve results further, we could: incorporate collaborative filtering signals (user-user or item-item similarities), use matrix factorization or neural embeddings instead of TF-IDF, tune the like_threshold and prop_weight parameters more carefully, add more sophisticated text preprocessing (lemmatization, n-grams), or ensemble multiple models together. For cold-start users, gathering a few initial preferences through a brief onboarding survey could significantly boost personalization.

## Example Recommendations

This cell generates personalized top-10 hotel recommendations for a sample user from the training data. The recommendations are personalized using the Content-Based model if the user has past ratings, or popularity-based if the user is new (cold-start). Already-reviewed hotels are filtered out automatically.

In [9]:

some_user = str(train_df["user"].iloc[0]) if not train_df.empty else "ANY"
engine.recommend(user=some_user, n=10, filter_seen_items=True)


,item,score
0,Hotel_Review-g293962-d498698-Reviews-Renuka_Ci...,0.391731
1,Hotel_Review-g297550-d1591790-Reviews-Cleopatr...,0.386066
2,Hotel_Review-g670039-d1931078-Reviews-Melia_Bu...,0.375779
3,Hotel_Review-g187262-d265109-Reviews-Hotel_New...,0.371769
4,Hotel_Review-g3363990-d649293-Reviews-The_Sain...,0.366485
5,Hotel_Review-g294217-d8316832-Reviews-The_Perk...,0.361686
6,Hotel_Review-g775984-d499983-Reviews-Hotel_Car...,0.360835
7,Hotel_Review-g293919-d1119639-Reviews-Ibis_Pat...,0.357273
8,Hotel_Review-g297698-d572368-Reviews-Kayumanis...,0.353232
